## **8th Annual WiDS Datathon Challenges:** 

### **Unraveling the Mysteries of the Female Brain**

# Data Understanding & Profiling

* [1. Introduction](#1-introduction)
* [2. Data Import](#2-data-import)
* [3. Dataset Schema](#3-dataset-schema)
* [4. Data Types & Basic Stats](#4-data-types--basic-stats)
* [5. Target Variables](#5-target-variables)
* [6. Feature Descriptions](#6-feature-descriptions)
* [7. Data Quality Summary](#7-data-quality-summary)

# 1. Introduction

https://www.kaggle.com/competitions/widsdatathon2025

The WiDS Datathon was developed with Ann S. Bowers Women's Brain Health Initiative (WBHI) in collaboration with Cornell University and UC Santa Barbara.

Datasets and support are provided by the Healthy Brain Network (HBN), the signature scientific initiative of the Child Mind Institute, and the Reproducible Brain Charts project (RBC).

We will analyze diagnostic data, socio-demographic, emotions, and parenting data, and functional MRI data from the Healthy Brain Network (HBN).

---

**Goal:** Build a model to predict both an individual's **sex** and their **ADHD diagnosis** using:
- Functional brain imaging data (fMRI connectome matrices)
- Socio-demographic, emotions, and parenting information

---

**Challenge Question:**

*"What brain activity patterns are associated with ADHD; are they different between males and females, and, if so, how?"*

# 2. Data Import

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

print(f"python version: {sys.version}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"matplotlib version: {matplotlib.__version__}")
print(f"seaborn version: {sns.__version__}")

In [ ]:
# data paths
file_path_train_cat_md  = "../data/raw/TRAIN_NEW/TRAIN_CATEGORICAL_METADATA_new.xlsx"
file_path_train_fun_cm  = "../data/raw/TRAIN_NEW/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv"
file_path_train_quan_md = "../data/raw/TRAIN_NEW/TRAIN_QUANTITATIVE_METADATA_new.xlsx"
file_path_train_sol     = "../data/raw/TRAIN_NEW/TRAINING_SOLUTIONS.xlsx"

In [ ]:
# loading data
train_cat  = pd.read_excel(file_path_train_cat_md)
train_fcm  = pd.read_csv(file_path_train_fun_cm)
train_quan = pd.read_excel(file_path_train_quan_md)
train_sol  = pd.read_excel(file_path_train_sol)

# 3. Dataset Schema

| File | Description |
|------|-------------|
| `TRAIN_CATEGORICAL_METADATA` | Demographics, ethnicity, parenting education/occupation, scan location |
| `TRAIN_QUANTITATIVE_METADATA` | Behavioral scores: EHQ, APQ, SDQ, ColorVision, Age at Scan |
| `TRAIN_FUNCTIONAL_CONNECTOME_MATRICES` | 200×200 Pearson correlation matrix per participant (19,900 features) |
| `TRAINING_SOLUTIONS` | Target labels: `ADHD_Outcome` and `Sex_F` |

In [ ]:
print("Record counts per file:")
print(f"  TRAIN_CATEGORICAL_METADATA  : {train_cat.shape[0]} rows, {train_cat.shape[1]} cols")
print(f"  TRAIN_FUNCTIONAL_CONNECTOME : {train_fcm.shape[0]} rows, {train_fcm.shape[1]} cols")
print(f"  TRAIN_QUANTITATIVE_METADATA : {train_quan.shape[0]} rows, {train_quan.shape[1]} cols")
print(f"  TRAINING_SOLUTIONS          : {train_sol.shape[0]} rows, {train_sol.shape[1]} cols")

# 4. Data Types & Basic Stats

In [ ]:
print("\n--- TRAIN_CATEGORICAL_METADATA ---")
train_cat.info()
train_cat.head()

In [ ]:
print("\n--- TRAIN_QUANTITATIVE_METADATA ---")
train_quan.info()
train_quan.describe()

In [ ]:
print("\n--- TRAINING_SOLUTIONS ---")
train_sol.info()
train_sol.describe()

In [ ]:
print("\n--- FUNCTIONAL CONNECTOME (first 5 cols) ---")
train_fcm.iloc[:, :6].head()

# 5. Target Variables

In [ ]:
print("Target distribution:")
print(train_sol[['ADHD_Outcome', 'Sex_F']].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

train_sol['ADHD_Outcome'].value_counts().plot(
    kind='bar', ax=axes[0], color=['green', 'orange'],
    edgecolor='black', linewidth=0.7
)
axes[0].set_title('ADHD Outcome')
axes[0].set_xticklabels(['No ADHD (0)', 'ADHD (1)'], rotation=0)
axes[0].set_ylabel('Count')

train_sol['Sex_F'].value_counts().plot(
    kind='bar', ax=axes[1], color=['blue', 'red'],
    edgecolor='black', linewidth=0.7
)
axes[1].set_title('Sex (Female = 1)')
axes[1].set_xticklabels(['Male (0)', 'Female (1)'], rotation=0)
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

⚠️ **Dataset is imbalanced.** Significantly more males than females, and more non-ADHD than ADHD participants. This must be handled during modeling (e.g. class weights, SMOTE, stratified splits).

# 6. Feature Descriptions

### Categorical Metadata
| Column | Description |
|--------|-------------|
| `participant_id` | Unique identifier |
| `Basic_Demos_Enroll_Year` | Year of enrollment |
| `Basic_Demos_Study_Site` | Study location code |
| `PreInt_Demos_Fam_Child_Ethnicity` | Child ethnicity |
| `PreInt_Demos_Fam_Child_Race` | Child race |
| `MRI_Track_Scan_Location` | Scan site |
| `Barratt_Barratt_P1_Edu` | Parent 1 education level |
| `Barratt_Barratt_P1_Occ` | Parent 1 occupation |
| `Barratt_Barratt_P2_Edu` | Parent 2 education level |
| `Barratt_Barratt_P2_Occ` | Parent 2 occupation |

### Quantitative Metadata
| Column Group | Description |
|-------------|-------------|
| `EHQ_EHQ_Total` | Handedness score |
| `ColorVision_CV_Score` | Color vision test score |
| `APQ_P_*` | Alabama Parenting Questionnaire (6 subscales) |
| `SDQ_SDQ_*` | Strengths & Difficulties Questionnaire (9 subscales) |
| `MRI_Track_Age_at_Scan` | Age at MRI scan (~30% missing) |

In [ ]:
# Quick feature inventory
print("Categorical metadata columns:")
print(train_cat.columns.tolist())
print("\nQuantitative metadata columns:")
print(train_quan.columns.tolist())

# 7. Data Quality Summary

In [ ]:
def missing_report(df, name):
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    report = pd.DataFrame({'missing_count': missing, 'missing_pct': pct})
    report = report[report['missing_count'] > 0].sort_values('missing_pct', ascending=False)
    print(f"\n=== {name} ===")
    if report.empty:
        print("✅ No missing values")
    else:
        print(report)

missing_report(train_cat,  "CATEGORICAL METADATA")
missing_report(train_quan, "QUANTITATIVE METADATA")
missing_report(train_sol,  "TRAINING SOLUTIONS")

fcm_missing = train_fcm.isnull().values.any()
print(f"\n=== FUNCTIONAL CONNECTOME ===")
print(f"{'❌ Has missing values' if fcm_missing else '✅ No missing values'}")

### Missing Value Summary

| Dataset | Status | Key Missing Columns |
|---------|--------|---------------------|
| Categorical Metadata | ❌ Has nulls | Ethnicity, Race, Scan Location, Parent Education/Occupation |
| Quantitative Metadata | ❌ Has nulls | `MRI_Track_Age_at_Scan` (~30%), `EHQ_Total`, `ColorVision`, APQ & SDQ (1 participant each) |
| Training Solutions | ✅ Clean | — |
| Functional Connectome | ✅ Clean | — |

**Recommended actions:**
- Impute or drop `MRI_Track_Age_at_Scan` (high missingness)
- Consider dropping Barratt (parenting SES) features with high null rates
- APQ and SDQ single-participant nulls: use median imputation